In [8]:
# SCD TYPE 1 — Üzerine Yazma (Overwrite)
# Senaryo: Müşteri telefon numarası değişti
# Eski veri kaybolur — geçmiş önemli değil

from pyspark.sql.functions import when, col

# Mevcut dim_customer tablosunu oku
df_customer = spark.read.table("dim_customer")

# Kaç müşteri var görelim
print(f"Toplam müşteri sayısı: {df_customer.count()}")

# Dragon Souveniers müşterisini bulalım
df_customer.filter(
    col("CUSTOMERNAME") == "Dragon Souveniers, Ltd."
).select("CUSTOMERNAME", "PHONE").show()

StatementMeta(, ad6449f9-6790-4fbd-a2d9-1523d06b57f0, 10, Finished, Cancelled, Cancelled, False)

In [ ]:
# SCD TYPE 1 — Telefon numarasını güncelle
# Eski numara: +65 221 7555
# Yeni numara: +65 221 9999
# Eski veri kaybolacak — Type 1 böyle çalışır!

from pyspark.sql.functions import when, col, lit

# Güncelleme yap
# when: eğer CUSTOMERNAME Dragon Souveniers ise → yeni numara yaz
# otherwise: diğer müşterilerin numarası değişmez
df_customer_updated = df_customer.withColumn(
    "PHONE",
    when(
        col("CUSTOMERNAME") == "Dragon Souveniers, Ltd.",
        lit("+65 221 9999")  # Yeni numara
    ).otherwise(col("PHONE"))  # Diğerleri değişmez
)

# Güncellenen veriyi Delta tablosuna kaydet — overwrite!
df_customer_updated.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("dim_customer")

# Doğrula — yeni numara geldi mi?
spark.read.table("dim_customer")\
    .filter(col("CUSTOMERNAME") == "Dragon Souveniers, Ltd.")\
    .select("CUSTOMERNAME", "PHONE")\
    .show()

print("✅ SCD Type 1 tamamlandı — eski numara silindi, yeni numara yazıldı!")

StatementMeta(, ad6449f9-6790-4fbd-a2d9-1523d06b57f0, -1, Cancelled, , Cancelled, True)

In [ ]:
# SCD TYPE 2 — Tarihsel Kayıt Tutma
# Eski veri silinmez, is_current=False yapılır
# Yeni kayıt eklenir, is_current=True

from pyspark.sql.functions import lit, current_date, when, col
from pyspark.sql.types import BooleanType, DateType

# Mevcut dim_customer'ı oku
df_scd2 = spark.read.table("dim_customer")

# SCD2 için gerekli kolonları ekle
# is_current: bu kayıt güncel mi?
# start_date: kaydın geçerlilik başlangıcı
# end_date: kaydın geçerlilik sonu (NULL = hala aktif)
df_scd2 = df_scd2\
    .withColumn("is_current", lit(True).cast(BooleanType()))\
    .withColumn("start_date", current_date())\
    .withColumn("end_date", lit(None).cast(DateType()))

# Mevcut tabloyu SCD2 formatında kaydet
df_scd2.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("dim_customer_scd2")

print(f"dim_customer_scd2 oluşturuldu: {df_scd2.count()} satır")
display(df_scd2.select(
    "CUSTOMERNAME", "PHONE", 
    "is_current", "start_date", "end_date"
).limit(5))

StatementMeta(, ad6449f9-6790-4fbd-a2d9-1523d06b57f0, -1, Cancelled, , Cancelled, True)

In [ ]:
# SCD TYPE 2 — Güncelleme Simülasyonu
# Dragon Souveniers şehir değiştirdi
# Eski kayıt: is_current=False, end_date=bugün
# Yeni kayıt: is_current=True, start_date=bugün

from pyspark.sql.functions import lit, current_date, when, col
from pyspark.sql.types import BooleanType, DateType

# Mevcut SCD2 tablosunu oku
df_scd2 = spark.read.table("dim_customer_scd2")

# ADIM 1: Eski Dragon Souveniers kaydını kapat
# is_current=False yap, end_date=bugün
df_updated = df_scd2.withColumn(
    "is_current",
    when(
        col("CUSTOMERNAME") == "Dragon Souveniers, Ltd.",
        lit(False).cast(BooleanType())
    ).otherwise(col("is_current"))
).withColumn(
    "end_date",
    when(
        col("CUSTOMERNAME") == "Dragon Souveniers, Ltd.",
        current_date()
    ).otherwise(col("end_date"))
)

# ADIM 2: Yeni Dragon Souveniers kaydı oluştur
# Singapur → Malezya, yeni telefon
from pyspark.sql import Row
new_row = df_scd2.filter(
    col("CUSTOMERNAME") == "Dragon Souveniers, Ltd."
).withColumn("PHONE", lit("+60 3 2999 8888"))\
 .withColumn("is_current", lit(True).cast(BooleanType()))\
 .withColumn("start_date", current_date())\
 .withColumn("end_date", lit(None).cast(DateType()))

# ADIM 3: Eski + yeni kayıtları birleştir
df_final = df_updated.union(new_row)

# Kaydet
df_final.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("dim_customer_scd2")

# Doğrula — Dragon Souveniers için 2 kayıt olmalı
spark.read.table("dim_customer_scd2")\
    .filter(col("CUSTOMERNAME") == "Dragon Souveniers, Ltd.")\
    .select("CUSTOMERNAME", "PHONE", "is_current", "start_date", "end_date")\
    .show()

print("✅ SCD Type 2 tamamlandı — tarihsel kayıt korundu!")

StatementMeta(, ad6449f9-6790-4fbd-a2d9-1523d06b57f0, -1, Cancelled, , Cancelled, True)

In [ ]:
# WINDOW FUNCTIONS — Gerçek Dünya Data Engineering
# fact_sales tablosundan ülke bazında ranking yapacağız

from pyspark.sql.functions import rank, lag, sum, col, round
from pyspark.sql.window import Window

# fact_sales tablosunu oku
df_fact = spark.read.table("fact_sales")
df_geo = spark.read.table("dim_geography")

# fact + geography join
df_joined = df_fact.join(
    df_geo,
    df_fact.COUNTRY == df_geo.COUNTRY,
    "left"
).select(
    df_fact.ORDERNUMBER,
    df_fact.SALES,
    df_fact.ORDERDATE,
    df_geo.COUNTRY,
    df_geo.TERRITORY
)

print(f"Joined tablo: {df_joined.count()} satır")
display(df_joined.limit(5)) #you see this is a many to many problem here 

StatementMeta(, ad6449f9-6790-4fbd-a2d9-1523d06b57f0, -1, Cancelled, , Cancelled, True)

In [ ]:
# Sorun: dim_geography'de aynı ülke birden fazla satırda var
# Çözüm: Sadece ülke bazında distinct al

from pyspark.sql.functions import rank, lag, sum, col, round
from pyspark.sql.window import Window

# fact_sales tablosunu oku
df_fact = spark.read.table("fact_sales")

# Ülke bazında toplam satış hesapla
df_country_sales = df_fact.groupBy("COUNTRY")\
    .agg(
        round(sum("SALES"), 2).alias("TOTAL_SALES"),
        sum("QUANTITYORDERED").alias("TOTAL_QTY")
    )

# Window: ülkeleri satışa göre sırala
window_rank = Window.orderBy(col("TOTAL_SALES").desc())

# RANK ekle
df_ranked = df_country_sales\
    .withColumn("SALES_RANK", rank().over(window_rank))

# Sonucu göster
display(df_ranked.orderBy("SALES_RANK"))
print(f"Toplam ülke sayısı: {df_ranked.count()}")

StatementMeta(, ad6449f9-6790-4fbd-a2d9-1523d06b57f0, -1, Cancelled, , Cancelled, True)

In [ ]:
# LAG FUNCTION — Bir önceki aya göre değişim
# "Geçen aya göre satışım arttı mı, azaldı mı?"

from pyspark.sql.functions import lag, round, sum, col, concat, lit
from pyspark.sql.window import Window

# fact_sales'ten ay bazında satış
df_monthly = spark.read.table("fact_sales")\
    .join(
        spark.read.table("dim_date").select("ORDERDATE", "MONTH_ID", "YEAR_ID"),
        "ORDERDATE"
    )\
    .groupBy("YEAR_ID", "MONTH_ID")\
    .agg(round(sum("SALES"), 2).alias("MONTHLY_SALES"))\
    .orderBy("YEAR_ID", "MONTH_ID")

# Window: yıl içinde aya göre sırala
window_lag = Window.partitionBy("YEAR_ID").orderBy("MONTH_ID")

# LAG: bir önceki ayın satışını getir
df_monthly = df_monthly\
    .withColumn("PREV_MONTH_SALES", lag("MONTHLY_SALES", 1).over(window_lag))\
    .withColumn(
        "MOM_CHANGE",  # Month over Month değişim
        round(
            (col("MONTHLY_SALES") - col("PREV_MONTH_SALES")) / 
            col("PREV_MONTH_SALES") * 100, 2
        )
    )

display(df_monthly)
print("✅ LAG function tamamlandı!")

StatementMeta(, ad6449f9-6790-4fbd-a2d9-1523d06b57f0, -1, Cancelled, , Cancelled, True)

In [ ]:
# Window function sonuçlarını Gold katmanına kaydet
# Bu tablo direkt Power BI'a bağlanabilir

# Rank tablosunu kaydet
df_ranked.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("gold_country_ranking")

# Monthly trend tablosunu kaydet
df_monthly.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("gold_monthly_trend")

print("✅ gold_country_ranking kaydedildi!")
print("✅ gold_monthly_trend kaydedildi!")

StatementMeta(, ad6449f9-6790-4fbd-a2d9-1523d06b57f0, -1, Cancelled, , Cancelled, True)

In [ ]:
# dim_geography duplicate sorunu düzeltme
# Sorun: Aynı ülke birden fazla şehirde var
# Çözüm: Ülke bazında DISTINCT tablo oluştur
from pyspark.sql.functions import monotonically_increasing_id

# Silver tablosundan sadece ülke bazında unique kayıtlar al
dim_geography_fixed = spark.read.table("silver_sales")\
    .select("COUNTRY", "TERRITORY")\
    .dropDuplicates()\
    .withColumn("geography_key", monotonically_increasing_id())

# Kaç unique ülke var?
print(f"Unique ülke sayısı: {dim_geography_fixed.count()}")

# Kaydet — overwrite ile eski tabloyu güncelle
dim_geography_fixed.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("dim_geography_country")

display(dim_geography_fixed)

StatementMeta(, ad6449f9-6790-4fbd-a2d9-1523d06b57f0, -1, Cancelled, , Cancelled, True)

In [ ]:
# Sadece COUNTRY bazında unique — TERRITORY'yi çıkar
from pyspark.sql.functions import monotonically_increasing_id, first

dim_geography_fixed_v2 = spark.read.table("silver_sales")\
    .select("COUNTRY", "TERRITORY")\
    .groupBy("COUNTRY")\
    .agg(first("TERRITORY").alias("TERRITORY"))\
    .withColumn("geography_key", monotonically_increasing_id())

print(f"Unique ülke sayısı: {dim_geography_fixed_v2.count()}")

dim_geography_fixed_v2.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("dim_geography_country")

display(dim_geography_fixed_v2)

StatementMeta(, ad6449f9-6790-4fbd-a2d9-1523d06b57f0, -1, Cancelled, , Cancelled, True)

In [ ]:
from pyspark.sql.functions import when, col

dim_geography_final = dim_geography_fixed_v2.withColumn(
    "TERRITORY",
    when(col("COUNTRY") == "SINGAPORE", "APAC")
    .otherwise(col("TERRITORY"))
)

# Kaydet
dim_geography_final.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("dim_geography_country")

print("Singapore → APAC olarak düzeltildi ✅")
display(dim_geography_final)

StatementMeta(, ad6449f9-6790-4fbd-a2d9-1523d06b57f0, -1, Cancelled, , Cancelled, True)

In [ ]:
# Hafta 3 tamamlanan tablolar
print("=== HAFTA 3 — MILESTONE 2 ÖZET ===")
print(f"dim_customer_scd2: {spark.read.table('dim_customer_scd2').count()} kayıt")
print(f"gold_country_ranking: {spark.read.table('gold_country_ranking').count()} kayıt")
print(f"gold_monthly_trend: {spark.read.table('gold_monthly_trend').count()} kayıt")
print(f"dim_geography_country: {spark.read.table('dim_geography_country').count()} kayıt")
print("=== TÜM TABLOLAR HAZIR ✅ ===")

StatementMeta(, ad6449f9-6790-4fbd-a2d9-1523d06b57f0, -1, Cancelled, , Cancelled, True)

In [ ]:
%run "./optimization_notebook"